# Custom Quantized LLM for OmniBrowser Agent

This notebook fine-tunes a small planner model in Colab (QLoRA), runs a sanity check on merged output, then quantizes to MLC format for WebLLM (q4f16_1).

Expected dataset format (JSONL):
- messages[0]: system prompt
- messages[1]: user message (prompt.ts style)
- messages[2]: assistant JSON string with {evaluation, memory, nextGoal, action}

In [ ]:
# Install a Colab-safe dependency set.
# If imports fail after experimentation, do Runtime -> Factory reset runtime and rerun from top.

!pip install -q -U \
  "transformers==4.46.3" \
  "trl==0.11.4" \
  "peft==0.13.2" \
  "accelerate==1.1.1" \
  "datasets==3.1.0"

# bitsandbytes 0.44.0 avoids triton.ops issues in newer Colab runtimes.
!pip install -q --no-deps "bitsandbytes==0.44.0"

In [ ]:
import importlib.util
from pathlib import Path

def patch_bnb_if_needed():
    try:
        import triton.ops.matmul_perf_model  # old API exists
        print("triton.ops exists; no patch needed")
        return
    except Exception:
        pass

    spec = importlib.util.find_spec("bitsandbytes")
    if spec is None:
        raise RuntimeError("bitsandbytes not installed")

    bnb_dir = Path(spec.origin).parent
    patches = {
        "triton/int8_matmul_mixed_dequantize.py":
            "import torch\n\ndef int8_matmul_mixed_dequantize(a, b, state_x, state_w, bias):\n    return None\n",
        "triton/int8_matmul_rowwise_dequantize.py":
            "import torch\n\ndef int8_matmul_rowwise_dequantize(a, b, state_x, state_w, bias):\n    return None\n",
    }
    for rel, content in patches.items():
        p = bnb_dir / rel
        if p.exists():
            p.write_text(content, encoding="utf-8")
            print("patched", p)

patch_bnb_if_needed()


In [ ]:
# Version sanity check
import torch, transformers, trl, peft, datasets, bitsandbytes
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
print("transformers", transformers.__version__)
print("trl", trl.__version__)
print("peft", peft.__version__)
print("datasets", datasets.__version__)
print("bitsandbytes", bitsandbytes.__version__)

In [ ]:
import os
import json
import torch
from datasets import load_dataset
from huggingface_hub import notebook_login
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
DATASET_PATH = "/content/omnibrowser_planner_train.jsonl"
OUTPUT_DIR = "/content/omnibrowser-planner-1p5b-lora"
MERGED_DIR = "/content/omnibrowser-planner-1p5b-merged"
MAX_SEQ_LEN = 2048
EPOCHS = 3
BATCH_SIZE = 2
GRAD_ACCUM = 8
LR = 2e-4
SEED = 42
VAL_SPLIT = 0.1

HF_MERGED_REPO = "akshayram1/omnibrowser-planner-1p5b"
HF_QUANT_REPO = "akshayram1/omnibrowser-planner-q4f16_1-MLC"
PUSH_TO_HF = False

print("dataset exists:", os.path.exists(DATASET_PATH))
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

In [ ]:
# Hugging Face login (required only if PUSH_TO_HF=True).
notebook_login()

In [ ]:
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
split = dataset.train_test_split(test_size=VAL_SPLIT, seed=SEED, shuffle=True)
raw_train = split["train"]
raw_eval = split["test"]

print("Total rows:", len(dataset))
print("Train rows:", len(raw_train))
print("Eval rows:", len(raw_eval))
print(raw_train[0]["messages"][0]["role"], raw_train[0]["messages"][1]["role"], raw_train[0]["messages"][2]["role"])
print(raw_train[0]["messages"][2]["content"][:200])

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "up_proj", "down_proj", "gate_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
def to_text(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

train_dataset = raw_train.map(to_text, remove_columns=raw_train.column_names)
eval_dataset = raw_eval.map(to_text, remove_columns=raw_eval.column_names)

print(train_dataset[0]["text"][:400])
print("Train/Eval text rows:", len(train_dataset), len(eval_dataset))

In [ ]:
base_args = dict(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=torch.cuda.is_available(),
    fp16=not torch.cuda.is_available(),
    max_seq_length=MAX_SEQ_LEN,
    report_to="none",
    seed=SEED,
)

try:
    training_args = SFTConfig(eval_strategy="epoch", **base_args)
except TypeError:
    training_args = SFTConfig(evaluation_strategy="epoch", **base_args)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    args=training_args,
)

trainer.train()
trainer.evaluate()

In [ ]:
# Merge LoRA into a full model
merged_model = trainer.model.merge_and_unload()
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print("Saved merged model to", MERGED_DIR)

if PUSH_TO_HF:
    merged_model.push_to_hub(HF_MERGED_REPO, private=True)
    tokenizer.push_to_hub(HF_MERGED_REPO, private=True)
    print("Pushed merged model to", HF_MERGED_REPO)

In [ ]:
# Post-training sanity check before quantization
check_model = AutoModelForCausalLM.from_pretrained(
    MERGED_DIR,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
check_tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR, use_fast=True)

sample_messages = raw_eval[0]["messages"][:2]
prompt = check_tokenizer.apply_chat_template(sample_messages, tokenize=False, add_generation_prompt=True)
inputs = check_tokenizer(prompt, return_tensors="pt").to(check_model.device)

with torch.no_grad():
    out = check_model.generate(**inputs, max_new_tokens=280, do_sample=False, temperature=0.0)

full_text = check_tokenizer.decode(out[0], skip_special_tokens=True)
generated = full_text[len(prompt):].strip() if full_text.startswith(prompt) else full_text
print("Raw generated output:\n", generated[:1200])

def extract_first_json(text):
    start = text.find("{")
    if start < 0:
        return None
    depth = 0
    in_string = False
    escaped = False
    for i in range(start, len(text)):
        ch = text[i]
        if escaped:
            escaped = False
            continue
        if ch == \"\\\" and in_string:
            escaped = True
            continue
        if ch == '\"':
            in_string = not in_string
            continue
        if in_string:
            continue
        if ch == '{':
            depth += 1
        elif ch == '}':
            depth -= 1
            if depth == 0:
                return text[start:i+1]
    return None

json_blob = extract_first_json(generated)
if json_blob is None:
    raise ValueError("Sanity check failed: no JSON object found in generated output")

parsed = json.loads(json_blob)
for key in ["evaluation", "memory", "nextGoal", "action"]:
    if key not in parsed:
        raise ValueError(f"Sanity check failed: missing key {key}")

action_type = parsed.get("action", {}).get("type")
if action_type not in {"click","type","navigate","extract","scroll","focus","wait","done"}:
    raise ValueError(f"Sanity check failed: invalid action.type {action_type}")

print("Sanity check passed. action.type =", action_type)

## Quantize to MLC format (q4f16_1)

For Qwen2.5, use a Qwen-family conversation template if available. The cell tries: qwen2 -> qwen -> chatml.

In [ ]:
!pip install -q -U mlc-llm mlc-ai-nightly -f https://mlc.ai/wheels

In [ ]:
import subprocess

MLC_OUT_DIR = "/content/dist/omnibrowser-planner-q4f16_1-MLC"

subprocess.run([
    "mlc_llm", "convert_weight", MERGED_DIR,
    "--quantization", "q4f16_1",
    "-o", MLC_OUT_DIR
], check=True)

template_candidates = ["qwen2", "qwen", "chatml"]
chosen_template = None
for template in template_candidates:
    proc = subprocess.run([
        "mlc_llm", "gen_config", MERGED_DIR,
        "--quantization", "q4f16_1",
        "--conv-template", template,
        "-o", MLC_OUT_DIR
    ], capture_output=True, text=True)

    if proc.returncode == 0:
        chosen_template = template
        print("gen_config succeeded with template:", template)
        break

    print(f"gen_config failed for template {template}:")
    print(proc.stderr[:500])

if chosen_template is None:
    raise RuntimeError("All conv-template attempts failed (qwen2/qwen/chatml).")

print("Chosen conv-template:", chosen_template)
subprocess.run(["ls", "-lah", MLC_OUT_DIR], check=True)

In [ ]:
# Upload quantized files
from huggingface_hub import HfApi

if PUSH_TO_HF:
    api = HfApi()
    api.create_repo(HF_QUANT_REPO, repo_type="model", private=True, exist_ok=True)
    api.upload_folder(
        repo_id=HF_QUANT_REPO,
        folder_path=MLC_OUT_DIR,
        repo_type="model",
    )
    print("Uploaded quantized files to", HF_QUANT_REPO)
else:
    print("PUSH_TO_HF is False; skipping upload.")

## WebLLM Integration

Use your uploaded quantized model from Hugging Face:

- model: https://huggingface.co/akshayram1/omnibrowser-planner-q4f16_1-MLC
- model_id: omnibrowser-planner-q4f16_1
- model_lib: webllm.modelLibURLPrefix + webllm.modelVersion + "/Qwen2.5-1.5B-Instruct-q4f16_1-ctx4k_cs1k-webgpu.wasm"